# Stage 3: Singlish Normalisation

### Project Structure

```bash
Latent/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb          
    ├─ Stage_2_POS_and_NER_Tagging.ipynb                        
-----------------------------------------------------------------            
│   ├─ Stage_3_Singlish_Normalisation.ipynb                     │
-----------------------------------------------------------------
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb tqdm

### Change Directory

In [ ]:
# Path
# ----
FOLDER_PATH: str = "/content/drive/MyDrive/Latent/"

import os
os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import gc
import regex
import spacy
import import_ipynb
import pandas as pd
from tqdm import tqdm
from Stage_2_POS_and_NER_Tagging import pos_tagger
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, save_single_df_to_csv, output_base_dir

## Singlish Normalisation

### POS Tagging Test

In [ ]:
sentences = [
    "Christian Ho sets sights on F3 after winning race",
    "trash to treasure what would you do with an essence of chicken box make chairs lor",
    "Sian la work tomorrow...",
    "k pop group le sserafim learn kopi ordering vocabulary for s pore concert",
    "how can lei",
    "ba in product design or ba in industrial design",
    "m sian man brings s 300 00 into s pore to gamble doesn t declare it because it s troublesome fined s 10 00"
]

for i, sent in enumerate(sentences):
    tags = pos_tagger.tag_text(sent)
    print(f"\nSentence {i+1}: {sent}")
    print("Tokens and tags:")
    for token_info in tags:
        print(f"  {token_info['token']:15} POS: {token_info['pos']:6} Tag: {token_info['tag']}")


In [ ]:
%cd /content/drive/MyDrive/Latent/data/

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/Latent/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/Latent/data/CommentVault.csv', keep_default_na=False)

### Singlish Normalization with POS

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
# Enable the tagger pipeline component, which assigns part-of-speech tags to tokens.
nlp.enable_pipe("tagger")

regex_map = []

mapping_file = '/content/drive/MyDrive/Latent/utilities/singlish_regex_to_text.txt'
with open(mapping_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip() 
        if not line or line.startswith('#'):
            continue
        parts = line.rsplit(',', 1)
        if len(parts) != 2:
            print("Skipping invalid line:", line)
            continue
        pattern, replacement = parts
        regex_map.append((pattern.strip(), replacement.strip()))

regex_map_compiled = [(regex.compile(pat, flags=regex.IGNORECASE), repl) for pat, repl in regex_map]

def normalize_token(word, pos=None):
    if pos == "PROPN":
        return word
    new_word = word
    for pattern, replacement in regex_map_compiled:
        new_word = pattern.sub(replacement, new_word)
    return new_word

def normalize_doc(doc):
    text = doc.text

    for tok in reversed(list(doc)):
        if tok.is_space or tok.pos_ == "PROPN":
            continue
        old = tok.text 
        new = normalize_token(old, tok.pos_) 
        
        if old != new:
            start, end = tok.idx, tok.idx + len(tok.text)
            text = text[:start] + new + text[end:]
    return text

def normalize_columns(df, text_columns, nlp, normalize_doc, batch_size=32, verbose=True):
    for col in text_columns:
        if col not in df.columns:
            continue

        normalized_col = f"singlish_normalized_{col.replace('cleaned_', '')}"

        texts = df[col].astype(str).tolist()
        normalized_texts = [] 

        for j in range(0, len(texts), batch_size):
            batch_texts = texts[j:j + batch_size]

            docs = list(nlp.pipe(batch_texts, batch_size=batch_size))
            batch_normalized = [normalize_doc(doc) for doc in docs]
            normalized_texts.extend(batch_normalized) 

            del docs
            del batch_normalized
            gc.collect()

        df[normalized_col] = normalized_texts

        if verbose:
            original_col_for_compare = df[col].astype(str).fillna('')
            normalized_col_for_compare = df[normalized_col].astype(str).fillna('')

            changed_rows = df[
                original_col_for_compare.str.replace(" ", "", regex=False) !=
                normalized_col_for_compare.str.replace(" ", "", regex=False)
            ]

            for idx, before, after in zip(
                changed_rows.index,
                changed_rows[col],
                changed_rows[normalized_col]
            ):
                print(f"ROW    : {idx}")
                print("BEFORE :", before)
                print("AFTER  :", after)
                print("-" * 50)

        del texts
        del normalized_texts
        gc.collect()

    return df

#### PostVault

In [ ]:
text_columns = ['cleaned_title', 'cleaned_content']

df_posts = normalize_columns(df_posts, text_columns, nlp, normalize_doc, batch_size=32, verbose=True)


#### CommentVault

In [ ]:
text_columns = ['cleaned_text']

df_comments = normalize_columns(df_comments, text_columns, nlp, normalize_doc, batch_size=32, verbose=True)

### Save Files

#### PostVault

In [ ]:
df_posts.columns

In [ ]:
# ---------
# PostVault
# ---------

display(df_posts.head())
print("-----------")
print("Posts Shape")
print("-----------")
print(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/Latent/data/", "PostVault")

#### CommentVault

In [ ]:
# ------------
# CommentVault
# ------------

display(df_comments.head())
print("--------------")
print("Comments Shape")
print("--------------")
print(df_comments.shape)
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/Latent/data/", "CommentVault")